# E3 — LSTM / GRU trên chuỗi ảnh

**CO5085 · Học sâu và ứng dụng trong thị giác máy tính · HK261** — tập dữ liệu: Fashion-MNIST (cùng split với E1, E2).

Notebook này tự chứa toàn bộ code của E3 và chạy được từ đầu đến cuối (khoảng 5 phút trên GPU T4).

| Yêu cầu của đề | Phần trong notebook |
|---|---|
| 1. LSTM và GRU (`nn.LSTM`, `nn.GRU`) cho phân loại | `RNNClassifier` |
| 2. Cách đưa ảnh thành chuỗi (≥ 2 cách) | theo hàng (28 bước) và theo patch 7×7 (16 bước) |
| 3. Baseline không hồi quy MLP/CNN | MLP, CNN của E1 (dùng lại checkpoint hoặc huấn luyện lại cùng split) |
| 4. So sánh accuracy, số tham số, khả năng bắt phụ thuộc thời gian | Mục *Kết quả* và *Nhìn một phần ảnh* |

## Thiết lập
Chạy lần lượt từ trên xuống. Trên Colab: *Runtime → Change runtime type → T4 GPU*.

In [ ]:
import csv
import json
import math
import os
import random
import time
from dataclasses import asdict, dataclass, field

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| thiết bị:",
      torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU (sẽ chậm, nên bật GPU)")

In [ ]:
# ---- hằng số và hàm vẽ/đọc kết quả (không cần PyTorch)
from matplotlib.ticker import MaxNLocator

CLASSES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
           "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
SHORT = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Boot"]
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
INK2, MUTED, GRID = "#52514e", "#8a8984", "#e4e3df"

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "legend.frameon": False, "lines.linewidth": 1.8, "savefig.bbox": "tight",
})


def load_run(run_dir):
    """Đọc lại một lần chạy đã lưu: summary.json, history.csv, logits trên tập test."""
    with open(os.path.join(run_dir, "summary.json"), encoding="utf-8") as f:
        summary = json.load(f)
    run = {"summary": summary, "history": pd.read_csv(os.path.join(run_dir, "history.csv"))}
    npz = os.path.join(run_dir, "test_outputs.npz")
    if os.path.isfile(npz):
        d = np.load(npz)
        run["logits"], run["targets"] = d["logits"], d["targets"]
        run["preds"] = run["logits"].argmax(1)
    return run


def summary_row(label, run):
    s = run["summary"]
    return {"Mô hình": label, "Số tham số": s["params"],
            "Train acc (%)": round(100 * s["final_train_acc"], 2),
            "Val acc tốt nhất (%)": round(100 * s["best_val_acc"], 2), "Epoch tốt nhất": s["best_epoch"],
            "Test acc (%)": round(100 * s["test_acc"], 2), "Test loss": round(s["test_loss"], 4),
            "Giây/epoch": round(s["mean_epoch_time"], 2), "Suy luận test (s)": round(s["test_inference_time"], 3)}


def plot_curves(runs, labels, colors, styles=None, save_as=None):
    """Loss và accuracy theo epoch. Nét chấm: train; nét liền/gạch: validation."""
    styles = styles or {}
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
    for key, run in runs.items():
        h, c, ls = run["history"], colors[key], styles.get(key, "-")
        ax1.plot(h["epoch"], h["train_loss"], color=c, ls=":", lw=1.3)
        ax1.plot(h["epoch"], h["val_loss"], color=c, ls=ls)
        ax2.plot(h["epoch"], 100 * h["train_acc"], color=c, ls=":", lw=1.3)
        ax2.plot(h["epoch"], 100 * h["val_acc"], color=c, ls=ls, label=labels[key])
    ax1.set(xlabel="epoch", ylabel="cross-entropy loss", title="Loss")
    ax2.set(xlabel="epoch", ylabel="accuracy (%)", title="Accuracy")
    ax2.legend(fontsize=8)
    for ax in (ax1, ax2):
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    fig.text(0.5, -0.02, "nét chấm: train · nét liền/gạch: validation", ha="center", color=INK2)
    fig.tight_layout()
    if save_as:
        fig.savefig(save_as)
    plt.show()


def confusion(targets, preds, k=10):
    cm = np.zeros((k, k), dtype=int)
    np.add.at(cm, (targets, preds), 1)
    return cm


def softmax_np(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

In [ ]:
EXERCISE = "E3"
SEED = 42
DATA_ROOT = "data"
RESULTS = "results/E3"
FIGURES = "figures/E3"
BATCH_SIZE = 128
QUICK = False          # True: mỗi mô hình chỉ 1 epoch, để chạy thử toàn bộ notebook thật nhanh
                       # (chạy thử xong nhớ xóa thư mục results/ trước khi chạy thật)
EPOCHS = 1 if QUICK else 15
LR = 1e-3
WEIGHT_DECAY = 1e-4
HIDDEN, LAYERS, RNN_DROPOUT, GRAD_CLIP = 128, 2, 0.2, 1.0
RUNS = ["lstm:rows", "gru:rows", "lstm:patch7", "gru:patch7"]   # cell:cách biểu diễn
E1_RESULTS = "results/E1"     # baseline MLP/CNN: dùng lại nếu đã chạy E1, nếu chưa thì huấn luyện tại đây
CELL_LABEL = {"lstm": "LSTM", "gru": "GRU"}
SEQ_LABEL = {"rows": "theo hàng", "cols": "theo cột", "patch7": "patch 7×7", "patch4": "patch 4×4"}
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(FIGURES, exist_ok=True)

### Seed, đo thời gian, lưu file

In [ ]:
def set_seed(seed: int = 42, deterministic: bool = True) -> None:
    """Cố định mọi nguồn ngẫu nhiên để kết quả có thể tái lập."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def count_params(model: torch.nn.Module, trainable_only: bool = True) -> int:
    """Tổng số phần tử của các tham số (p.numel())."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad or not trainable_only)


def sync(device: torch.device) -> None:
    """Chờ GPU chạy xong trước khi đọc đồng hồ, nếu không thời gian đo sẽ sai."""
    if device.type == "cuda":
        torch.cuda.synchronize()


class Timer:
    """Dùng với `with Timer(device) as t: ...` rồi đọc t.elapsed (giây)."""

    def __init__(self, device: torch.device):
        self.device = device
        self.elapsed = 0.0

    def __enter__(self):
        sync(self.device)
        self._start = time.perf_counter()
        return self

    def __exit__(self, *exc):
        sync(self.device)
        self.elapsed = time.perf_counter() - self._start
        return False


def save_json(obj, path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def load_json(path: str):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def environment_info() -> dict:
    """Ghi lại môi trường chạy để đưa vào báo cáo."""
    info = {
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    }
    try:
        import torchvision

        info["torchvision"] = torchvision.__version__
    except ImportError:
        pass
    return info

## Dữ liệu: Fashion-MNIST

- 60 000 ảnh train gốc được hoán vị bằng seed cố định: **54 000 train**, **6 000 validation**.
  10 000 ảnh test gốc chỉ dùng cho kết quả cuối.
- Ảnh đưa về $[0,1]$ rồi chuẩn hóa bằng mean/std tính **chỉ trên phần train**. Không tăng cường dữ liệu.
- Toàn bộ ảnh được giữ sẵn dưới dạng tensor trong bộ nhớ (28×28 nên rất nhẹ), mỗi epoch không phải đọc lại từ đĩa.
- Cả ba bài dùng đúng cùng split này (cùng seed), nên kết quả so sánh được với nhau.

In [ ]:
CLASSES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]
NUM_CLASSES = 10
IMAGE_SHAPE = (1, 28, 28)


@dataclass
class DataBundle:
    train: DataLoader
    val: DataLoader
    test: DataLoader
    mean: float
    std: float
    sizes: dict


def _load_raw(root: str):
    from torchvision import datasets

    train = datasets.FashionMNIST(root, train=True, download=True)
    test = datasets.FashionMNIST(root, train=False, download=True)
    # .data: uint8 (N, 28, 28), .targets: int64 (N,)
    return train.data, train.targets, test.data, test.targets


def split_indices(n_total: int, val_size: int, seed: int):
    """Trả về (train_idx, val_idx) từ một hoán vị cố định."""
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(n_total, generator=g)
    return perm[val_size:], perm[:val_size]


def get_data(root: str = "data", batch_size: int = 128, val_size: int = 6000,
             split_seed: int = 42, loader_seed: int = 0, eval_batch_size: int = 1000) -> DataBundle:
    x_tr_all, y_tr_all, x_te, y_te = _load_raw(root)
    train_idx, val_idx = split_indices(len(x_tr_all), val_size, split_seed)

    x_tr = x_tr_all[train_idx].float().div_(255.0)
    x_va = x_tr_all[val_idx].float().div_(255.0)
    x_te = x_te.float().div_(255.0)

    mean, std = x_tr.mean().item(), x_tr.std().item()

    def prep(x):  # (N, 28, 28) -> (N, 1, 28, 28), chuẩn hóa theo thống kê của tập train
        return ((x - mean) / std).unsqueeze(1)

    ds_tr = TensorDataset(prep(x_tr), y_tr_all[train_idx])
    ds_va = TensorDataset(prep(x_va), y_tr_all[val_idx])
    ds_te = TensorDataset(prep(x_te), y_te)

    # Generator riêng cho DataLoader: cùng loader_seed -> cùng thứ tự batch giữa các lần chạy.
    g = torch.Generator().manual_seed(loader_seed)
    pin = torch.cuda.is_available()
    train_loader = DataLoader(ds_tr, batch_size=batch_size, shuffle=True, generator=g, pin_memory=pin)
    val_loader = DataLoader(ds_va, batch_size=eval_batch_size, shuffle=False, pin_memory=pin)
    test_loader = DataLoader(ds_te, batch_size=eval_batch_size, shuffle=False, pin_memory=pin)

    sizes = {"train": len(ds_tr), "val": len(ds_va), "test": len(ds_te)}
    return DataBundle(train_loader, val_loader, test_loader, mean, std, sizes)


def black_value(mean: float, std: float) -> float:
    """Giá trị của một pixel đen (0) sau khi chuẩn hóa; dùng để che một phần ảnh ở E3."""
    return (0.0 - mean) / std

In [ ]:
data = get_data(DATA_ROOT, batch_size=BATCH_SIZE, loader_seed=SEED)
print("Kích thước:", data.sizes, "| mean = %.4f, std = %.4f" % (data.mean, data.std))

# Ảnh test gốc (uint8, chưa chuẩn hóa) để hiển thị
TEST_IMAGES = datasets.FashionMNIST(DATA_ROOT, train=False, download=True).data.numpy()
TEST_LABELS = datasets.FashionMNIST(DATA_ROOT, train=False, download=True).targets.numpy()

fig, axes = plt.subplots(2, 10, figsize=(12, 2.8))
for c in range(10):
    idx = np.where(TEST_LABELS == c)[0][:2]
    for r in range(2):
        axes[r, c].imshow(TEST_IMAGES[idx[r]], cmap="gray_r")
        axes[r, c].axis("off")
    axes[0, c].set_title(CLASSES[c], fontsize=8)
plt.suptitle("Hai ảnh mẫu của mỗi lớp")
plt.show()

## Vòng lặp huấn luyện tự viết

Mỗi bước đi đúng trình tự **forward → loss → `backward()` → `optimizer.step()`**, không dùng `trainer.fit()`.

- **Loss:** `nn.CrossEntropyLoss` nhận logits (đã gồm log-softmax), nên mô hình không đặt softmax ở cuối `forward`.
- **Optimizer:** AdamW, không áp weight decay cho bias, tham số chuẩn hóa, token CLS và positional embedding.
- **Learning rate:** đổi theo từng bước, warmup tuyến tính (nếu có) rồi giảm theo cosine về 0.
- **Chọn mô hình:** sau mỗi epoch đánh giá trên validation, giữ checkpoint có val accuracy cao nhất;
  tập test chỉ được đánh giá một lần bằng checkpoint đó.
- **Lưu kết quả:** mỗi lần chạy ghi `history.csv`, `summary.json`, `best.pt`, `test_outputs.npz`
  (cùng định dạng với script trong repo, nên `scripts/make_report_assets.py` dùng lại được).

In [ ]:
@dataclass
class TrainConfig:
    epochs: int = 15
    lr: float = 1e-3
    weight_decay: float = 1e-4
    warmup_epochs: float = 0.0      # số epoch tăng LR tuyến tính từ 0 trước khi giảm theo cosine
    grad_clip: float = 0.0          # 0 = không cắt gradient
    label_smoothing: float = 0.0
    seed: int = 42
    extra: dict = field(default_factory=dict)  # thông tin riêng của từng bài (kiến trúc, tokenizer...)


def make_optimizer(model: nn.Module, cfg: TrainConfig) -> torch.optim.Optimizer:
    """AdamW; không áp weight decay lên bias, LayerNorm/BatchNorm, CLS token và positional embedding."""
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim <= 1 or name.endswith(".bias") or "pos_embed" in name or "cls_token" in name:
            no_decay.append(p)
        else:
            decay.append(p)
    groups = [
        {"params": decay, "weight_decay": cfg.weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ]
    groups = [g for g in groups if g["params"]]
    return torch.optim.AdamW(groups, lr=cfg.lr)


def make_scheduler(optimizer, cfg: TrainConfig, steps_per_epoch: int):
    """LR theo từng bước: warmup tuyến tính rồi giảm theo cosine về 0."""
    total = cfg.epochs * steps_per_epoch
    warmup = int(cfg.warmup_epochs * steps_per_epoch)

    def lr_lambda(step: int) -> float:
        if warmup > 0 and step < warmup:
            return (step + 1) / warmup
        progress = (step - warmup) / max(1, total - warmup)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train_one_epoch(model, loader, optimizer, scheduler, criterion, device, grad_clip=0.0):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)                    # 1. forward: (B, 10) logits, CHƯA qua softmax
        loss = criterion(logits, y)          # 2. loss: CrossEntropyLoss = log-softmax + NLL

        optimizer.zero_grad(set_to_none=True)
        loss.backward()                      # 3. backward: tính gradient
        if grad_clip > 0:
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()                     # 4. cập nhật trọng số
        scheduler.step()                     # LR thay đổi theo từng bước

        bs = y.size(0)
        total_loss += loss.item() * bs
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += bs
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, criterion, device, return_outputs=False):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    all_logits, all_targets = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        total_loss += criterion(logits, y).item() * y.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += y.size(0)
        if return_outputs:
            all_logits.append(logits.float().cpu())
            all_targets.append(y.cpu())
    if return_outputs:
        return total_loss / n, correct / n, torch.cat(all_logits), torch.cat(all_targets)
    return total_loss / n, correct / n


def fit(model: nn.Module, data, cfg: TrainConfig, device, out_dir: str, name: str) -> dict:
    """Huấn luyện, chọn checkpoint tốt nhất theo val accuracy, đánh giá trên test, lưu log.

    Ghi ra out_dir:
        history.csv   - loss/acc train & val, LR, thời gian từng epoch
        best.pt       - trọng số có val accuracy cao nhất
        test_outputs.npz - logits và nhãn trên tập test (để vẽ confusion matrix, tìm mẫu sai)
        summary.json  - tóm tắt số liệu + cấu hình + môi trường
    """
    os.makedirs(out_dir, exist_ok=True)
    model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    optimizer = make_optimizer(model, cfg)
    scheduler = make_scheduler(optimizer, cfg, len(data.train))

    n_params = count_params(model)
    print(f"[{name}] params = {n_params:,} | device = {device}")

    history = []
    best_val_acc, best_epoch = -1.0, -1
    ckpt_path = os.path.join(out_dir, "best.pt")
    for epoch in range(1, cfg.epochs + 1):
        lr_now = optimizer.param_groups[0]["lr"]
        with Timer(device) as t:
            tr_loss, tr_acc = train_one_epoch(model, data.train, optimizer, scheduler,
                                              criterion, device, cfg.grad_clip)
        va_loss, va_acc = evaluate(model, data.val, criterion, device)
        history.append({
            "epoch": epoch, "train_loss": tr_loss, "train_acc": tr_acc,
            "val_loss": va_loss, "val_acc": va_acc, "lr": lr_now, "epoch_time": t.elapsed,
        })
        if va_acc > best_val_acc:
            best_val_acc, best_epoch = va_acc, epoch
            torch.save(model.state_dict(), ckpt_path)
        print(f"[{name}] epoch {epoch:2d}/{cfg.epochs} | train loss {tr_loss:.4f} acc {tr_acc:.4f} "
              f"| val loss {va_loss:.4f} acc {va_acc:.4f} | {t.elapsed:.1f}s")

    with open(os.path.join(out_dir, "history.csv"), "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
        writer.writeheader()
        writer.writerows(history)

    # Đánh giá trên test bằng checkpoint tốt nhất theo val (không chọn theo test).
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    te_loss, te_acc, logits, targets = evaluate(model, data.test, criterion, device, return_outputs=True)
    np.savez_compressed(os.path.join(out_dir, "test_outputs.npz"),
                        logits=logits.numpy().astype(np.float32), targets=targets.numpy())

    # Thời gian suy luận trên toàn bộ tập test (sau 1 lượt chạy làm nóng).
    evaluate(model, data.test, criterion, device)
    with Timer(device) as t_inf:
        evaluate(model, data.test, criterion, device)

    epoch_times = [h["epoch_time"] for h in history]
    summary = {
        "name": name,
        "params": n_params,
        "best_epoch": best_epoch,
        "best_val_acc": best_val_acc,
        "test_acc": te_acc,
        "test_loss": te_loss,
        "final_train_acc": history[-1]["train_acc"],
        "final_train_loss": history[-1]["train_loss"],
        "mean_epoch_time": float(np.mean(epoch_times[1:] if len(epoch_times) > 1 else epoch_times)),
        "total_train_time": float(np.sum(epoch_times)),
        "test_inference_time": t_inf.elapsed,
        "data_sizes": data.sizes,
        "norm_mean": data.mean,
        "norm_std": data.std,
        "config": asdict(cfg),
        "env": environment_info(),
    }
    save_json(summary, os.path.join(out_dir, "summary.json"))
    print(f"[{name}] best val acc {best_val_acc:.4f} (epoch {best_epoch}) | test acc {te_acc:.4f}")
    return summary

## 1. Biến ảnh thành chuỗi

- **Theo hàng (`rows`):** bước thời gian $t$ là hàng thứ $t$, đọc từ trên xuống — $T=28$ bước, mỗi bước $F=28$ giá trị.
- **Theo patch (`patch7`):** lưới 4×4 patch 7×7, duyệt theo hàng (trái → phải, trên → dưới) — $T=16$, $F=49$.

Hai cách đánh đổi độ dài chuỗi với lượng thông tin mỗi bước.

In [ ]:
def to_sequence(x: torch.Tensor, mode: str) -> torch.Tensor:
    """(B, C, H, W) -> (B, T, F)."""
    B, C, H, W = x.shape
    if mode == "rows":
        return x.permute(0, 2, 1, 3).reshape(B, H, C * W)
    if mode == "cols":
        return x.permute(0, 3, 1, 2).reshape(B, W, C * H)
    if mode.startswith("patch"):
        p = int(mode[len("patch"):])
        x = x.unfold(2, p, p).unfold(3, p, p)            # (B, C, H/p, W/p, p, p)
        x = x.permute(0, 2, 3, 1, 4, 5)                   # (B, H/p, W/p, C, p, p)
        return x.reshape(B, (H // p) * (W // p), C * p * p)
    raise ValueError(f"mode không hợp lệ: {mode}")


def sequence_shape(mode: str, C: int = 1, H: int = 28, W: int = 28):
    """(T, F) của từng cách biểu diễn."""
    if mode == "rows":
        return H, C * W
    if mode == "cols":
        return W, C * H
    if mode.startswith("patch"):
        p = int(mode[len("patch"):])
        return (H // p) * (W // p), C * p * p
    raise ValueError(mode)


class RNNClassifier(nn.Module):
    def __init__(self, cell: str = "lstm", seq_mode: str = "rows", hidden: int = 128,
                 num_layers: int = 2, dropout: float = 0.2, num_classes: int = 10):
        super().__init__()
        self.cell, self.seq_mode = cell, seq_mode
        T, F = sequence_shape(seq_mode)
        self.seq_len, self.input_size = T, F
        rnn_cls = {"lstm": nn.LSTM, "gru": nn.GRU}[cell]
        self.rnn = rnn_cls(input_size=F, hidden_size=hidden, num_layers=num_layers,
                           batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.head = nn.Linear(hidden, num_classes)

    def forward(self, x):                       # (B, 1, 28, 28)
        seq = to_sequence(x, self.seq_mode)     # (B, T, F)
        out, _ = self.rnn(seq)                  # (B, T, hidden): output tầng trên cùng ở mọi bước
        return self.head(out[:, -1])            # dùng bước cuối -> (B, 10)

    @torch.no_grad()
    def logits_every_step(self, x):
        """Logits nếu dừng ở từng bước t = 1..T: (B, T, 10).

        Vì RNN một chiều, output ở bước t chỉ phụ thuộc t phần đầu của chuỗi, nên đây
        chính là dự đoán khi mô hình mới "nhìn thấy" t hàng/patch đầu tiên.
        """
        out, _ = self.rnn(to_sequence(x, self.seq_mode))
        return self.head(out)


def rnn_param_formula(cell: str, input_size: int, hidden: int, num_layers: int) -> int:
    """Số tham số lý thuyết của nn.LSTM / nn.GRU (PyTorch dùng 2 vector bias mỗi cổng).
    Mỗi cổng ở tầng l: W_ih (H x in_l) + W_hh (H x H) + b_ih (H) + b_hh (H).
    LSTM có 4 cổng (i, f, g, o), GRU có 3 cổng (r, z, n)."""
    gates = {"lstm": 4, "gru": 3}[cell]
    total, in_l = 0, input_size
    for _ in range(num_layers):
        total += gates * (hidden * in_l + hidden * hidden + 2 * hidden)
        in_l = hidden
    return total

In [ ]:
img = datasets.FashionMNIST(DATA_ROOT, train=False, download=True).data[:1].float().unsqueeze(1) / 255.0
seq_rows = to_sequence(img, "rows")[0].numpy()        # (28, 28)
seq_patch = to_sequence(img, "patch7")[0].numpy()     # (16, 49)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), gridspec_kw={"width_ratios": [1, 1, 1.9]})
axes[0].imshow(img[0, 0].numpy(), cmap="gray_r")
axes[0].set_title("ảnh gốc 28×28")
axes[1].imshow(seq_rows, cmap="gray_r", aspect="auto")
axes[1].set(title="theo hàng: T=28, F=28", xlabel="đặc trưng F", ylabel="bước t")
axes[2].imshow(seq_patch, cmap="gray_r", aspect="auto")
axes[2].set(title="theo patch 7×7: T=16, F=49", xlabel="đặc trưng F (7×7 pixel)", ylabel="bước t")
for ax in axes:
    ax.grid(False)
plt.tight_layout()
plt.show()

## 2. Mô hình LSTM / GRU

chuỗi `(B, T, F)` → `nn.LSTM` hoặc `nn.GRU` (2 tầng, hidden 128, dropout 0.2 giữa các tầng, `batch_first=True`)
→ hidden của **bước cuối** ở tầng trên cùng → `Linear(128, 10)`.

Số tham số tầng đầu: $g\,(HF + H^2 + 2H)$ với $g=4$ cổng cho LSTM (input, forget, cell, output) và $g=3$ cho GRU
(reset, update, new) — cùng $H$ thì GRU có đúng 3/4 tham số hồi quy của LSTM.

In [ ]:
rows = []
for run in RUNS:
    cell, mode = run.split(":")
    m = RNNClassifier(cell, mode, HIDDEN, LAYERS, RNN_DROPOUT)
    rows.append({"Mô hình": f"{CELL_LABEL[cell]} {SEQ_LABEL[mode]}", "T": m.seq_len, "F": m.input_size,
                 "Tham số RNN (đếm)": count_params(m.rnn),
                 "Tham số RNN (công thức)": rnn_param_formula(cell, m.input_size, HIDDEN, LAYERS),
                 "Tổng tham số": count_params(m)})
display(pd.DataFrame(rows).set_index("Mô hình"))

## 3. Baseline không hồi quy: MLP và CNN của E1

Nếu đã chạy notebook E1 (có `results/E1/mlp` và `results/E1/cnn`), dùng lại checkpoint. Nếu chưa, huấn luyện lại
đúng cấu hình E1 trên cùng split.

In [ ]:
IN_DIM = 28 * 28
NUM_CLASSES = 10


class SoftmaxClassifier(nn.Module):
    """Hồi quy softmax: flatten -> một ánh xạ tuyến tính W x + b -> (softmax trong loss)."""

    def __init__(self, in_dim: int = IN_DIM, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(in_dim, num_classes)

    def forward(self, x):               # x: (B, 1, 28, 28)
        return self.fc(self.flatten(x))  # (B, 10)


class MLPClassifier(nn.Module):
    """MLP trên vector flatten: 784 -> 512 -> 256 -> 10, ReLU + Dropout ở các tầng ẩn."""

    def __init__(self, in_dim: int = IN_DIM, hidden=(512, 256), num_classes: int = NUM_CLASSES,
                 dropout: float = 0.2):
        super().__init__()
        layers, prev = [nn.Flatten()], in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(inplace=True), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def conv_bn_relu(c_in: int, c_out: int) -> nn.Sequential:
    # bias=False vì BatchNorm ngay sau đã có tham số dịch (beta), bias của conv là thừa.
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(c_out),
        nn.ReLU(inplace=True),
    )


class CNNClassifier(nn.Module):
    """CNN nhỏ kiểu VGG cho ảnh 28x28:
        [conv-bn-relu x2, maxpool]  1  -> 32 kênh, 28x28 -> 14x14
        [conv-bn-relu x2, maxpool]  32 -> 64 kênh, 14x14 -> 7x7
        flatten (64*7*7 = 3136) -> FC 128 -> dropout -> FC 10
    """

    def __init__(self, in_ch: int = 1, channels=(32, 64), fc_dim: int = 128,
                 num_classes: int = NUM_CLASSES, dropout: float = 0.3):
        super().__init__()
        blocks, prev = [], in_ch
        for c in channels:
            blocks += [conv_bn_relu(prev, c), conv_bn_relu(c, c), nn.MaxPool2d(2)]
            prev = c
        self.features = nn.Sequential(*blocks)
        spatial = 28 // (2 ** len(channels))
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(prev * spatial * spatial, fc_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(fc_dim, num_classes),
        )

    def forward(self, x):
        return self.head(self.features(x))


MODELS = {
    "softmax": SoftmaxClassifier,
    "mlp": MLPClassifier,
    "cnn": CNNClassifier,
}


def build_model(name: str) -> nn.Module:
    return MODELS[name]()

In [ ]:
for name in ["mlp", "cnn"]:
    out_dir = os.path.join(E1_RESULTS, name)
    if os.path.isfile(os.path.join(out_dir, "best.pt")) and os.path.isfile(os.path.join(out_dir, "summary.json")):
        print(f"Dùng lại {out_dir}")
        continue
    set_seed(SEED)
    data = get_data(DATA_ROOT, batch_size=BATCH_SIZE, loader_seed=SEED)
    cfg = TrainConfig(epochs=1 if QUICK else 15, lr=1e-3, weight_decay=1e-4, seed=SEED,
                      extra={"exercise": "E1", "model": name, "batch_size": BATCH_SIZE})
    fit(build_model(name), data, cfg, device, out_dir, name=f"E1-{name}")

## 4. Huấn luyện LSTM / GRU
AdamW (LR 1e-3, weight decay 1e-4), cosine, cắt gradient theo chuẩn 1.0 để RNN ổn định.

In [ ]:
for run in RUNS:
    cell, mode = run.split(":")
    set_seed(SEED)
    data = get_data(DATA_ROOT, batch_size=BATCH_SIZE, loader_seed=SEED)
    model = RNNClassifier(cell, mode, HIDDEN, LAYERS, RNN_DROPOUT)
    cfg = TrainConfig(epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP, seed=SEED,
                      extra={"exercise": "E3", "cell": cell, "seq_mode": mode, "seq_len": model.seq_len,
                             "input_size": model.input_size, "hidden": HIDDEN, "layers": LAYERS,
                             "dropout": RNN_DROPOUT, "batch_size": BATCH_SIZE})
    fit(model, data, cfg, device, os.path.join(RESULTS, f"{cell}_{mode}"), name=f"E3-{cell}-{mode}")

## 5. Kết quả

In [ ]:
runs, labels, colors, styles = {}, {}, {}, {}
for run in RUNS:
    cell, mode = run.split(":")
    k = (cell, mode)
    runs[k] = load_run(os.path.join(RESULTS, f"{cell}_{mode}"))
    labels[k] = f"{CELL_LABEL[cell]} · {SEQ_LABEL[mode]}"
    colors[k] = SERIES[0] if cell == "lstm" else SERIES[1]
    styles[k] = "-" if mode == "rows" else "--"
for j, b in enumerate(["mlp", "cnn"]):
    runs[b] = load_run(os.path.join(E1_RESULTS, b))
    labels[b] = f"{b.upper()} (E1)"
    colors[b] = SERIES[2 + j]
    styles[b] = "-."

table = pd.DataFrame([summary_row(labels[k], r) for k, r in runs.items()]).set_index("Mô hình")
display(table)
table.to_csv(os.path.join(FIGURES, "e3_table.csv"))
plot_curves(runs, labels, colors, styles, save_as=os.path.join(FIGURES, "e3_curves.png"))

## 6. Nhìn một phần ảnh: khả năng tích lũy thông tin theo thời gian

- **LSTM/GRU:** RNN một chiều nên output ở bước $t$ chỉ phụ thuộc $t$ phần đầu. Đưa output đó qua tầng phân loại
  cho biết mô hình dự đoán thế nào khi mới "đọc" được một phần ảnh — không cần huấn luyện lại.
- **MLP/CNN:** giữ $k$ hàng đầu, thay các hàng còn lại bằng pixel đen.

Không mô hình nào được huấn luyện với ảnh bị che, nên đây là phép thử như nhau cho cả hai nhóm.

In [ ]:
@torch.no_grad()
def rnn_prefix_accuracy(model, loader):
    model.eval()
    correct, n = None, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model.logits_every_step(x)                       # (B, T, 10)
        hit = (logits.argmax(-1) == y[:, None]).sum(0).float()    # (T,)
        correct = hit if correct is None else correct + hit
        n += y.size(0)
    return (correct / n).cpu().tolist()


@torch.no_grad()
def masked_rows_accuracy(model, loader, fill, H=28):
    model.eval()
    correct, n = torch.zeros(H, device=device), 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        for k in range(1, H + 1):
            xm = x.clone()
            xm[:, :, k:, :] = fill                                # chỉ giữ k hàng đầu
            correct[k - 1] += (model(xm).argmax(1) == y).sum()
        n += y.size(0)
    return (correct / n).cpu().tolist()


data = get_data(DATA_ROOT, batch_size=BATCH_SIZE, loader_seed=SEED)
prefix = {}
for run in RUNS:
    cell, mode = run.split(":")
    model = RNNClassifier(cell, mode, HIDDEN, LAYERS, RNN_DROPOUT).to(device)
    model.load_state_dict(torch.load(os.path.join(RESULTS, f"{cell}_{mode}", "best.pt"), map_location=device))
    acc = rnn_prefix_accuracy(model, data.test)
    prefix[f"{cell}_{mode}"] = {"kind": "rnn", "fraction_seen": [(t + 1) / len(acc) for t in range(len(acc))],
                                "acc": acc}
for name in ["mlp", "cnn"]:
    model = build_model(name).to(device)
    model.load_state_dict(torch.load(os.path.join(E1_RESULTS, name, "best.pt"), map_location=device))
    acc = masked_rows_accuracy(model, data.test, black_value(data.mean, data.std))
    prefix[f"{name}_masked_rows"] = {"kind": "baseline", "fraction_seen": [(k + 1) / 28 for k in range(28)],
                                     "acc": acc}
save_json(prefix, os.path.join(RESULTS, "prefix_accuracy.json"))

In [ ]:
with open(os.path.join(RESULTS, "prefix_accuracy.json"), encoding="utf-8") as f:
    prefix = json.load(f)
fig, ax = plt.subplots(figsize=(7, 4.2))
summary = []
for name, d in prefix.items():
    if d["kind"] == "rnn":
        cell, mode = name.split("_", 1)
        c, ls = (SERIES[0] if cell == "lstm" else SERIES[1]), ("-" if mode == "rows" else "--")
        lab = f"{CELL_LABEL[cell]} · {SEQ_LABEL[mode]} (dừng ở bước t)"
    else:
        b = name.split("_")[0]
        c, ls, lab = (SERIES[2] if b == "mlp" else SERIES[3]), "-.", f"{b.upper()} (E1, che các hàng chưa thấy)"
    x, acc = 100 * np.array(d["fraction_seen"]), 100 * np.array(d["acc"])
    ax.plot(x, acc, color=c, ls=ls, label=lab)
    n = len(acc)
    summary.append({"Mô hình": lab, "25%": round(acc[n // 4 - 1], 1), "50%": round(acc[n // 2 - 1], 1),
                    "75%": round(acc[3 * n // 4 - 1], 1), "100%": round(acc[-1], 1)})
ax.axhline(10, color=MUTED, lw=0.8, ls=":")
ax.set(xlabel="phần trăm số pixel đã thấy (từ trên xuống)", ylabel="test accuracy (%)",
       title="Dự đoán khi mới thấy một phần ảnh", xlim=(0, 100), ylim=(0, 100))
ax.legend(fontsize=7, loc="upper left")
fig.savefig(os.path.join(FIGURES, "e3_prefix.png"))
plt.show()
display(pd.DataFrame(summary).set_index("Mô hình"))

## Ghi chú khi viết nhận xét

- LSTM vs GRU: accuracy, số tham số, thời gian; cách biểu diễn nào tốt hơn và vì sao.
- So với MLP/CNN: RNN đứng ở đâu; cấu trúc 1 chiều theo thời gian khác cấu trúc 2 chiều của CNN thế nào.
- Nhìn một phần ảnh: accuracy tăng thế nào theo số hàng đã đọc; RNN có dự đoán sớm tốt hơn không, vì sao.

## Lưu kết quả

In [ ]:
# Nén kết quả để tải về (Colab). Chạy trên máy cá nhân thì kết quả đã nằm sẵn trong thư mục results/ và figures/.
import shutil

archive = shutil.make_archive(f"{EXERCISE}_results", "zip", ".", base_dir="results")
shutil.make_archive(f"{EXERCISE}_figures", "zip", ".", base_dir="figures")
print("Đã tạo", archive, "và", f"{EXERCISE}_figures.zip")
try:
    from google.colab import files
    files.download(archive)
    files.download(f"{EXERCISE}_figures.zip")
except ImportError:
    pass